## Medallion Architecture Demo

In [78]:
# set up all parameters in this cell
products_file = 'abfss://raw@gwisynadls.dfs.core.windows.net/demo_training/products.csv'
dbname = 'gwinnett_lake'
tbl_bronze = 'products_bronze'
tbl_silver = 'products_silver'
tbl_gold = 'products_summary'
cat_table = 'dim_category'

StatementMeta(spark35, 5, 81, Finished, Available, Finished, False)

In [79]:
# create database if it does not exist and use it
spark.sql(f"create database if not exists {dbname};")
spark.sql(f"use {dbname};")

StatementMeta(spark35, 5, 82, Finished, Available, Finished, False)

DataFrame[]

In [80]:
# read the products.csv file in, but do not infer the schema
df = spark.read.format("csv").options(inferSchema=False, header=True).load(products_file)

StatementMeta(spark35, 5, 83, Finished, Available, Finished, False)

In [81]:
# create a SQL object so we can clean using SQL functions
df.createOrReplaceTempView("products_csv")

StatementMeta(spark35, 5, 84, Finished, Available, Finished, False)

In [82]:
# read into bronze dataframe
bronze_df = spark.sql(f"select *, current_timestamp as IngestedDateTimeUTC from products_csv;")

StatementMeta(spark35, 5, 85, Finished, Available, Finished, False)

In [83]:
# write to bronze
bronze_df.write.format("delta").mode("overwrite").saveAsTable(tbl_bronze)

StatementMeta(spark35, 5, 86, Finished, Available, Finished, False)

In [84]:
# create silver dataframe with product (cast types, deduplicate records)
products_silver = spark.sql(f"""
select distinct cast(ProductID as int) as ProductID, 
    trim(ProductName) as ProductName, 
    trim(Category) as Category, 
    round(cast(ListPrice as double),2) as ListPrice,
    round(cast(ListPrice as double) * 0.75,2) as Cost,
    IngestedDateTimeUTC
from {tbl_bronze}
    where ProductID is not null;
    """)

StatementMeta(spark35, 5, 87, Finished, Available, Finished, False)

In [85]:
# write silver to table
products_silver.write.format("delta").mode("overwrite").saveAsTable(tbl_silver)

StatementMeta(spark35, 5, 88, Finished, Available, Finished, False)

In [86]:
# compute a gold aggregate table
gold_product_summary = spark.sql(f"""
select 
    Category, 
    count(*) as NumProducts,
    round(avg(ListPrice),2) as AvgListPrice, 
    round(avg(ListPrice-Cost),2) as AvgMargin
from {tbl_silver}
    group by Category;
    """)

StatementMeta(spark35, 5, 89, Finished, Available, Finished, False)

In [87]:
# write gold summary table
gold_product_summary.write.format("delta").mode("overwrite").saveAsTable(tbl_gold)

StatementMeta(spark35, 5, 90, Finished, Available, Finished, False)

In [88]:
# build category table inside a dataframe using sql
category_df = spark.sql(f"""
select 
    DENSE_RANK() OVER (ORDER BY category) AS category_id,
    Category as category
from {tbl_silver}
group by Category;
""")

StatementMeta(spark35, 5, 91, Finished, Available, Finished, False)

In [89]:
# write category table
category_df.write.format("delta").mode("overwrite").saveAsTable(cat_table)

StatementMeta(spark35, 5, 92, Finished, Available, Finished, False)